In [ ]:
import sys

dbutils.widgets.text("project_root", "/Workspace/Shared/UberEats", "Workspace package root")
PROJECT_ROOT = dbutils.widgets.get("project_root").strip().rstrip("/")
if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql.functions import *
from pyspark.sql.types import *
from src.common.spark_utils import standardize_columns
from src.common.quality_events import log_pipeline_failure, quarantine_rejected_rows, reconcile_merge_target
from delta.tables import DeltaTable

In [ ]:
# Environment-specific storage configuration (override the widget per workspace/job).
dbutils.widgets.text("bronze_root", "abfss://bronze@ubereatsrisstorage.dfs.core.windows.net", "Bronze storage root")
BRONZE_ROOT = dbutils.widgets.get("bronze_root").rstrip("/")



# Read the Bronze source.
try:
    df_promotions_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{BRONZE_ROOT}/csv/promotions/")
    )
except Exception as exc:
    log_pipeline_failure(spark, "promotions", "source_read", exc)
    raise

source_rows = df_promotions_raw.count()
print(f"promotions.source_rows={source_rows}")
display(df_promotions_raw)
df_promotions_raw.printSchema()

In [ ]:
# Standardize column names.
df_promotions = standardize_columns(df_promotions_raw)

# Capture source quality metrics before deduplication and business filtering.
null_promo_ids = df_promotions.filter(col("promo_id").isNull()).count()
null_restaurant_ids = df_promotions.filter(col("restaurant_id").isNull()).count()
duplicate_promotion_keys = (
    df_promotions
    .filter(col("promo_id").isNotNull() & col("restaurant_id").isNotNull())
    .groupBy("promo_id", "restaurant_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
invalid_discount_values = (
    df_promotions
    .filter(col("discount_value").isNull() | (col("discount_value") <= 0))
    .count()
)
missing_promo_codes = df_promotions.filter(
    col("promo_code").isNull() | (trim(col("promo_code")) == "")
).count()

print(f"promotions.quality.null_promo_id_rows={null_promo_ids}")
print(f"promotions.quality.null_restaurant_id_rows={null_restaurant_ids}")
print(f"promotions.quality.duplicate_composite_keys={duplicate_promotion_keys}")

# No deterministic winner rule exists for duplicate composite keys. Quarantine
# every row in each duplicate group and stop rather than choosing arbitrarily.
duplicate_promotion_key_values = (
    df_promotions
    .filter(col("promo_id").isNotNull() & col("restaurant_id").isNotNull())
    .groupBy("promo_id", "restaurant_id").count().filter(col("count") > 1)
    .select("promo_id", "restaurant_id")
)
duplicate_promotion_rows = df_promotions.join(
    duplicate_promotion_key_values, on=["promo_id", "restaurant_id"], how="inner"
)
quarantine_rejected_rows(
    spark, duplicate_promotion_rows, "promotions",
    "duplicate_promo_restaurant_key_group_no_winner_policy",
)
if duplicate_promotion_keys:
    raise ValueError(
        f"Promotions Silver input contains {duplicate_promotion_keys} duplicate composite-key groups; "
        "all rows in those groups were quarantined and publishing was stopped"
    )
print(f"promotions.quality.invalid_discount_rows={invalid_discount_values}")
print(f"promotions.quality.missing_promo_code_rows={missing_promo_codes}")

# Quarantine invalid records before applying the existing positive-discount filter.
quarantine_run_id = str(__import__("uuid").uuid4())
invalid_business_keys = df_promotions.filter(
    col("promo_id").isNull() | col("restaurant_id").isNull()
)
quarantine_rejected_rows(
    spark, invalid_business_keys, "promotions",
    "missing_promo_id_or_restaurant_id", run_id=quarantine_run_id,
)
if null_promo_ids or null_restaurant_ids:
    raise ValueError(
        "Promotions Silver input contains rows with null merge keys; "
        "records were quarantined and publishing was stopped"
    )
quarantine_rejected_rows(
    spark,
    df_promotions.filter(col("discount_value").isNull() | (col("discount_value") <= 0)),
    "promotions", "discount_value_null_or_nonpositive", run_id=quarantine_run_id,
)

# Preserve the current behavior while exposing what is removed. The composite
# key matches the existing Delta MERGE condition.
df_promotions = df_promotions.drop_duplicates(["promo_id", "restaurant_id"])
df_promotions = df_promotions.filter(col("discount_value") > 0)

prepared_duplicate_keys = (
    df_promotions
    .filter(col("promo_id").isNotNull() & col("restaurant_id").isNotNull())
    .groupBy("promo_id", "restaurant_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
if prepared_duplicate_keys:
    raise ValueError(
        "Promotions Silver merge input contains "
        f"{prepared_duplicate_keys} duplicate (promo_id, restaurant_id) keys"
    )

print(f"promotions.quality.prepared_rows={df_promotions.count()}")
print(f"promotions.quality.prepared_duplicate_composite_keys={prepared_duplicate_keys}")
display(df_promotions)

In [ ]:
try:
    table_name = "ubereats_databricks1.silver.promotions"
    
    if not spark.catalog.tableExists(table_name):
        (
            df_promotions.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )
    else:
        target = DeltaTable.forName(spark, table_name)
        (
            target.alias("t")
            .merge(
                df_promotions.alias("s"),
                "t.promo_id = s.promo_id AND t.restaurant_id = s.restaurant_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
except Exception as exc:
    # Record publish failures without replacing the original exception.
    failure_record = [{
        "run_id": str(__import__("uuid").uuid4()),
        "dataset_name": "promotions",
        "stage": "silver_publish",
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:2000],
        "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
    }]
    try:
        spark.createDataFrame(failure_record).write.format("delta").mode("append").saveAsTable(
            "ubereats_databricks1.silver.pipeline_failure_events"
        )
    except Exception as log_exc:
        print("promotions.failure_log_error=" + str(log_exc)[:500])
    raise

# Reconcile the published Delta target before marking this run successful.
reconcile_merge_target(
    spark=spark,
    source_df=df_promotions,
    target_table=table_name,
    dataset_name="promotions",
    key_columns=["promo_id","restaurant_id"],
    compare_columns=["discount_value","promo_code"],
)

# Persist a compact successful-run quality record for later monitoring.
quality_table = "ubereats_databricks1.silver.pipeline_quality_results"
quality_record = [{
    "run_id": str(__import__("uuid").uuid4()),
    "dataset_name": "promotions",
    "source_rows": int(source_rows),
    "prepared_rows": int(df_promotions.count()),
    "target_rows_after": int(spark.table(table_name).count()),
    "status": "SUCCEEDED",
    "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
}]
spark.createDataFrame(quality_record).write.format("delta").mode("append").saveAsTable(quality_table)
print(f"promotions.quality.persisted_run_id=" + quality_record[0]["run_id"])